# LeetCode 3509 — Maximum Product of Subsequences With an Alternating Sum Equal to K bằng Python

- Link: https://leetcode.com/problems/maximum-product-of-subsequences-with-an-alternating-sum-equal-to-k/
- Độ khó: **Hard**
- Chủ đề: **DP trên dãy con** với trạng thái `(chẵn lẻ độ dài, tổng xen kẽ)` lưu **tập các tích đạt được**, chặn tích theo `limit`, xử lý đặc biệt số `0`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxProduct(self, nums: List[int], k: int, limit: int) -> int:
```

## 1. Mục tiêu học

1. Hiểu **tổng xen kẽ** của dãy con: phần tử ở vị trí chẵn (trong dãy con) mang dấu `+`, vị trí lẻ mang dấu `−` ⇒ khi thêm phần tử mới, dấu phụ thuộc **chẵn lẻ độ dài hiện tại**.
2. Nhận ra "tích lớn nhất **không vượt** `limit`" **không** có cấu trúc con tối ưu kiểu `max` ⇒ phải lưu **tập các tích** đạt được.
3. Chặn tích: mọi tích `> limit` gộp thành một trạng thái `OVER` — vẫn cần giữ vì nhân với `0` sẽ thành `0`.
4. Giới hạn miền tổng: `|tổng xen kẽ| ≤ Σ nums ≤ 1800` ⇒ nếu `|k|` lớn hơn thì trả `−1` ngay.
5. Kiểm chứng bằng vét cạn mọi dãy con.

## 2. Đề bài & ví dụ

Cho mảng `nums`, số `k` và `limit`. Tìm dãy con **khác rỗng** (giữ thứ tự) sao cho:

- tổng xen kẽ (`a0 − a1 + a2 − …`) bằng `k`;
- tích các phần tử **lớn nhất** nhưng **không vượt quá** `limit`.

Trả về tích đó, hoặc `−1` nếu không có dãy con nào thỏa.

**Ví dụ 1:** `nums=[1,2,3], k=2, limit=10` → `6` (`[1,2,3]`: `1−2+3 = 2`, tích 6; `[2]`: tích 2).

**Ví dụ 2:** `nums=[0,2,3], k=−5, limit=12` → `−1`.

**Ví dụ 3:** `nums=[2,2,3,3], k=0, limit=9` → `9` (`[2,2,3,3]` có tích 36 > 9; `[3,3]` tích 9).

**Ràng buộc:** `1 ≤ n ≤ 150`, `0 ≤ nums[i] ≤ 12`, `−10^5 ≤ k ≤ 10^5`, `1 ≤ limit ≤ 5000`.

In [1]:
from typing import List
from itertools import combinations
examples = [([1, 2, 3], 2, 10, 6), ([0, 2, 3], -5, 12, -1), ([2, 2, 3, 3], 0, 9, 9)]
for nums, k, lim, exp in examples:
    print(nums, k, lim, '-> kỳ vọng', exp)

[1, 2, 3] 2 10 -> kỳ vọng 6
[0, 2, 3] -5 12 -> kỳ vọng -1
[2, 2, 3, 3] 0 9 -> kỳ vọng 9


## 3. Kiến thức nền

### 3.1. Tổng xen kẽ

In [2]:
def alt_sum(seq):
    return sum(v if i % 2 == 0 else -v for i, v in enumerate(seq))

print(alt_sum([1, 2, 3]), alt_sum([2, 2, 3, 3]), alt_sum([5]))

2 0 5


### 3.2. Thêm phần tử vào cuối dãy con

Dãy con hiện có độ dài `L`. Thêm `x`: nếu `L` chẵn thì `x` ở vị trí chẵn ⇒ tổng `+ x`; nếu `L` lẻ thì `− x`. Vậy trạng thái cần biết **chẵn lẻ của độ dài** (không cần độ dài thật).

### 3.3. `set` và hợp tập

`A |= B` thêm mọi phần tử của `B` vào `A`. Dùng để gộp các tích đạt được.

In [3]:
A = {2, 6}
A |= {6, 9}
print(A, {p * 3 for p in A})

{9, 2, 6} {18, 27, 6}


### 3.4. Vì sao không thể chỉ lưu "tích lớn nhất"?

Với `limit = 9`: hai dãy con cùng trạng thái có tích 4 và 8. Lưu "lớn nhất" ⇒ giữ 8; sau đó nhân 2 ⇒ 16 > 9 bị loại; nhưng `4·2 = 8` lại hợp lệ. Mất nghiệm ⇒ phải giữ **tập** tích (≤ limit).

In [4]:
limit = 9
print('giữ max:', [p for p in [8 * 2] if p <= limit], '| giữ tập:', [p for p in [4 * 2, 8 * 2] if p <= limit])

giữ max: [] | giữ tập: [8]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Duyệt `nums` từ trái sang phải; mỗi phần tử có thể **bắt đầu** một dãy con mới hoặc **nối** vào một dãy con đã có. Trạng thái của một dãy con (cho tương lai) gồm:

- `p` = chẵn lẻ độ dài (quyết định dấu phần tử kế);
- `s` = tổng xen kẽ hiện tại;
- `prod` = tích hiện tại (chặn: `> limit` ⇒ `OVER`).

Tính tay ví dụ 3 (`[2,2,3,3]`, `limit = 9`), chỉ các trạng thái có `s = 0` ở cuối: `[2,2]` (tích 4), `[3,3]` (9), `[2,2,3,3]` (36 ⇒ OVER), `[2,3]`? tổng `−1` (không phải 0). Lớn nhất `≤ 9` là 9.

In [5]:
nums = [2, 2, 3, 3]
for r in range(1, 5):
    for idx in combinations(range(4), r):
        seq = [nums[i] for i in idx]
        if alt_sum(seq) == 0:
            p = 1
            for v in seq:
                p *= v
            print(seq, 'tích', p)

[2, 2] tích 4
[3, 3] tích 9
[2, 2, 3, 3] tích 36


## 5. Cách 1 — Brute Force: thử mọi dãy con

In [6]:
def max_product_brute(nums, k, limit):
    n, best = len(nums), -1
    for mask in range(1, 1 << n):
        seq = [nums[i] for i in range(n) if mask >> i & 1]
        if alt_sum(seq) == k:
            p = 1
            for v in seq:
                p *= v
            if p <= limit:
                best = max(best, p)
    return best

for nums, k, lim, exp in examples:
    print(max_product_brute(nums, k, lim), '| kỳ vọng', exp)

6 | kỳ vọng 6
-1 | kỳ vọng -1
9 | kỳ vọng 9


**Độ phức tạp:** `O(2^n · n)` — chỉ với `n ≤ 18`.

**Nút thắt:** nhiều dãy con khác nhau có cùng `(p, s, prod)` ⇒ gộp trạng thái.

## 6. Ý tưởng thuật toán / bước đột phá

**DP tập trạng thái:** `dp[(p, s)]` = tập các tích (đã chặn) của các dãy con khác rỗng có chẵn lẻ độ dài `p` và tổng xen kẽ `s`.

Khi xét phần tử `x`:

- **bắt đầu mới:** thêm `x` (chặn) vào `dp[(1, x)]`;
- **nối:** với mỗi `(p, s)` cũ, `s' = s + x` nếu `p = 0` ngược lại `s − x`; tập mới `{prod · x}` (chặn) thêm vào `dp[(1 − p, s')]`.

Các quan sát giúp chạy nhanh:

- `x = 0` ⇒ mọi tích thành `0` (chỉ thêm `{0}`); `x = 1` ⇒ tập giữ nguyên.
- Tích `≤ limit ≤ 5000` từ các số `≤ 12` chỉ có vài trăm giá trị khác nhau (số "12-mịn").
- `OVER` vẫn phải giữ: `OVER · 0 = 0`.

Đáp án: `max{prod ∈ dp[(0,k)] ∪ dp[(1,k)] : prod ≤ limit}` hoặc `−1`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP lưu **bộ ba** `(p, s, prod)` trong một tập phẳng

Cách trực quan nhất của DP trạng thái: một `set` các bộ ba. Đúng nhưng nhiều thao tác băm bộ.

In [7]:
def max_product_triples(nums, k, limit):
    if abs(k) > sum(nums):
        return -1
    OVER = limit + 1
    cap = lambda v: v if v <= limit else OVER
    states = set()
    for x in nums:
        new = {(1, x, cap(x))}
        for p, s, pr in states:
            ns = s + x if p == 0 else s - x
            new.add((1 - p, ns, 0 if x == 0 else cap(pr * x) if pr != OVER else OVER))
        states |= new
    best = -1
    for p, s, pr in states:
        if s == k and pr <= limit:
            best = max(best, pr)
    return best

for nums, k, lim, exp in examples:
    print(max_product_triples(nums, k, lim), '| kỳ vọng', exp)

6 | kỳ vọng 6
-1 | kỳ vọng -1
9 | kỳ vọng 9


Lưu ý `OVER · x` vẫn là `OVER` nếu `x ≥ 1`, và là `0` nếu `x = 0`.

### 7.2. Cách 3 — DP `dict[(p, s)] → set(tích)` (lời giải chính)

In [8]:
def max_product_dp(nums, k, limit):
    if abs(k) > sum(nums):
        return -1
    OVER = limit + 1
    dp = {}
    for x in nums:
        new = {}
        new[(1, x)] = {x if x <= limit else OVER}
        for (p, s), prods in dp.items():
            key = (1 - p, s + x if p == 0 else s - x)
            tgt = new.get(key)
            if tgt is None:
                tgt = new[key] = set()
            if x == 0:
                tgt.add(0)
            elif x == 1:
                tgt |= prods
            else:
                for pr in prods:
                    q = pr * x
                    tgt.add(q if q <= limit else OVER)
        for key, v in new.items():
            if key in dp:
                dp[key] |= v
            else:
                dp[key] = v
    best = -1
    for p in (0, 1):
        for pr in dp.get((p, k), ()):
            if pr <= limit and pr > best:
                best = pr
    return best

for nums, k, lim, exp in examples:
    print(max_product_dp(nums, k, lim), '| kỳ vọng', exp)

6 | kỳ vọng 6
-1 | kỳ vọng -1
9 | kỳ vọng 9


**Độ phức tạp:** gọi `S = Σ nums ≤ 1800`, `P` = số tích khác nhau `≤ limit` (vài trăm). Mỗi phần tử duyệt `O(2 · (2S+1) · P)` cặp trạng thái trong trường hợp xấu — thực tế ít hơn nhiều vì các trạng thái đạt được thưa.

### 7.3. Cách 4 — Tách riêng trường hợp tích bằng 0

Một tối ưu tư duy: nếu tồn tại dãy con **chứa số 0** có tổng xen kẽ `k`, đáp án ít nhất là 0. Phần còn lại chỉ cần DP với các tích **dương** (bỏ `OVER` vì tích dương vượt limit không bao giờ quay lại). Kiểm tra "có dãy con chứa 0 với tổng k" bằng DP tập tổng `(p, s, đã_có_0)`.

In [9]:
def max_product_split_zero(nums, k, limit):
    if abs(k) > sum(nums):
        return -1
    # (a) có dãy con chứa 0 với tổng xen kẽ k?
    reach = set()                                    # (p, s, has0)
    for x in nums:
        new = {(1, x, x == 0)}
        for p, s, z in reach:
            new.add((1 - p, s + x if p == 0 else s - x, z or x == 0))
        reach |= new
    zero_ok = any(s == k and z for p, s, z in reach)
    # (b) DP chỉ với tích dương <= limit (bỏ qua mọi dãy con chứa 0)
    dp = {}
    for x in nums:
        new = {}
        if x > 0 and x <= limit:
            new[(1, x)] = {x}
        if x > 0:
            for (p, s), prods in dp.items():
                key = (1 - p, s + x if p == 0 else s - x)
                tgt = new.setdefault(key, set())
                for pr in prods:
                    if pr * x <= limit:
                        tgt.add(pr * x)
        else:
            # x = 0: các dãy con cũ có thể "bỏ qua" 0 — không thay đổi dp
            pass
        for key, v in new.items():
            if v:
                dp.setdefault(key, set()).update(v)
    best = 0 if zero_ok else -1
    for p in (0, 1):
        for pr in dp.get((p, k), ()):
            best = max(best, pr)
    return best

for nums, k, lim, exp in examples:
    print(max_product_split_zero(nums, k, lim), '| kỳ vọng', exp)

6 | kỳ vọng 6
-1 | kỳ vọng -1
9 | kỳ vọng 9


Ở phần (b), dãy con không chứa 0 chỉ được tạo từ các phần tử dương; khi gặp `0`, các dãy con cũ đơn giản là **không chọn** nó — đúng ngữ nghĩa dãy con.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1 (`[1, 2, 3]`, `k = 2`, `limit = 10`): in `dp` sau mỗi phần tử.

In [10]:
def trace(nums, k, limit):
    OVER = limit + 1
    dp = {}
    for x in nums:
        new = {(1, x): {x if x <= limit else OVER}}
        for (p, s), prods in dp.items():
            key = (1 - p, s + x if p == 0 else s - x)
            tgt = new.setdefault(key, set())
            for pr in prods:
                q = 0 if x == 0 else min(pr * x, OVER) if pr != OVER else OVER
                tgt.add(q)
        for key, v in new.items():
            dp.setdefault(key, set()).update(v)
        shown = {f"(p={p}, s={s})": sorted(v) for (p, s), v in sorted(dp.items())}
        print(f"sau x={x}: {shown}")
    cand = sorted(dp.get((0, k), set()) | dp.get((1, k), set()))
    print(f"tích với tổng {k}: {cand} -> đáp án {max([c for c in cand if c <= limit], default=-1)}")

trace([1, 2, 3], 2, 10)

sau x=1: {'(p=1, s=1)': [1]}
sau x=2: {'(p=0, s=-1)': [2], '(p=1, s=1)': [1], '(p=1, s=2)': [2]}
sau x=3: {'(p=0, s=-2)': [3], '(p=0, s=-1)': [2, 6], '(p=1, s=1)': [1], '(p=1, s=2)': [2, 6], '(p=1, s=3)': [3]}
tích với tổng 2: [2, 6] -> đáp án 6


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def maxProduct(self, nums: List[int], k: int, limit: int) -> int:
        if abs(k) > sum(nums):
            return -1
        OVER = limit + 1
        dp = {}
        for x in nums:
            new = {(1, x): {x if x <= limit else OVER}}
            for (p, s), prods in dp.items():
                key = (1 - p, s + x if p == 0 else s - x)
                tgt = new.get(key)
                if tgt is None:
                    tgt = new[key] = set()
                if x == 0:
                    tgt.add(0)
                elif x == 1:
                    tgt |= prods
                else:
                    for pr in prods:
                        q = pr * x
                        tgt.add(q if q <= limit else OVER)
            for key, v in new.items():
                if key in dp:
                    dp[key] |= v
                else:
                    dp[key] = v
        best = -1
        for p in (0, 1):
            for pr in dp.get((p, k), ()):
                if best < pr <= limit:
                    best = pr
        return best

for nums, k, lim, exp in examples:
    got = Solution().maxProduct(nums, k, lim)
    print(got, 'OK' if got == exp else 'SAI')

6 OK
-1 OK
9 OK


## 10. Vì sao thuật toán đúng

1. **Trạng thái đủ:** tổng xen kẽ sau khi nối `x` chỉ phụ thuộc tổng hiện tại và chẵn lẻ độ dài; tích chỉ phụ thuộc tích hiện tại. Vậy `(p, s, prod)` mô tả đầy đủ "tương lai" của một dãy con.
2. **Bao phủ:** mọi dãy con khác rỗng được tạo đúng một lần theo phần tử cuối cùng của nó: bắt đầu mới tại phần tử đầu, nối dần các phần tử sau. Hợp tập giữ mọi trạng thái đạt được.
3. **Chặn tích an toàn:** với `x ≥ 1`, tích chỉ tăng hoặc giữ nguyên ⇒ một khi `> limit` thì mãi `> limit` trừ khi gặp `0`. Gộp mọi giá trị `> limit` thành `OVER` không làm mất thông tin cần thiết (`OVER · 0 = 0`, `OVER · x = OVER`).
4. **Miền tổng:** `|tổng xen kẽ| ≤ Σ nums` ⇒ `|k| > Σ nums` thì không có nghiệm.
5. **Không thể chỉ lưu max** (mục 3.4) ⇒ lưu tập là cần thiết.

## 11. Phân tích Time / Space Complexity

Gọi `n ≤ 150`, `S = Σ nums ≤ 1800`, `P` = số tích phân biệt `≤ limit` (≤ vài trăm).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^n n)` | `O(n)` |
| 2. Tập bộ ba | `O(n · 2(2S+1) · P)` xấu nhất | như trên |
| 3. `dict` → `set` | như trên, hằng số nhỏ hơn | `O(2(2S+1) · P)` |
| 4. Tách số 0 | như trên | như trên |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {'triples': max_product_triples, 'dp': max_product_dp, 'split_zero': max_product_split_zero,
         'submit': lambda a, k, l: Solution().maxProduct(a, k, l)}
edge = [
    ([0], 0, 1, 0),
    ([5], 5, 4, -1),            # tích 5 > limit
    ([12, 0], 12, 1, 0),        # [12, 0]: tổng 12 - 0 = 12, tích 0
    ([3, 3], 0, 10, 9),
    ([1] * 10, 1, 1, 1),
    ([7, 7], 100000, 5000, -1),
]
for nums, k, lim, exp in examples + edge:
    assert max_product_brute(nums, k, lim) == exp, (nums, k, lim)
    for name, f in impls.items():
        got = f(nums, k, lim)
        assert got == exp, (name, nums, k, lim, got, exp)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [13]:
import random
random.seed(3509)
cnt = 0
for _ in range(500):
    n = random.randint(1, 12)
    nums = [random.choice([0, 1, random.randint(0, 12)]) for _ in range(n)]
    k = random.randint(-15, 15)
    lim = random.randint(1, 400)
    exp = max_product_brute(nums, k, lim)
    for name, f in impls.items():
        assert f(nums, k, lim) == exp, (name, nums, k, lim)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test ngẫu nhiên.')

Khớp vét cạn trên 500 bộ test ngẫu nhiên.


### 12.3. Ca lớn (`n = 150`, `limit = 5000`)

In [14]:
import time
random.seed(150)
for trial in range(4):
    nums = [random.randint(0, 12) if trial < 3 else random.randint(1, 12) for _ in range(150)]
    k = random.randint(-30, 30)
    t = time.time(); a = Solution().maxProduct(nums, k, 5000); t1 = time.time() - t
    t = time.time(); b = max_product_split_zero(nums, k, 5000); t2 = time.time() - t
    assert a == b
    print(f"trial {trial}: k={k:>3} -> {a} | dict/set {t1:.2f}s | tách số 0 {t2:.2f}s")

trial 0: k= -7 -> 5000 | dict/set 0.29s | tách số 0 0.28s


trial 1: k= -9 -> 5000 | dict/set 0.30s | tách số 0 0.28s


trial 2: k=-13 -> 5000 | dict/set 0.27s | tách số 0 0.29s


trial 3: k=-28 -> 4860 | dict/set 0.31s | tách số 0 0.25s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ lưu tích **lớn nhất** `≤ limit` cho mỗi trạng thái

In [15]:
def wrong_keep_max(nums, k, limit):
    if abs(k) > sum(nums):
        return -1
    best_at = {}
    for x in nums:
        new = {}
        if x <= limit:
            new[(1, x)] = x
        for (p, s), pr in best_at.items():
            key = (1 - p, s + x if p == 0 else s - x)
            q = pr * x
            if q <= limit:
                new[key] = max(new.get(key, -1), q)
        for key, v in new.items():
            best_at[key] = max(best_at.get(key, -1), v)
    return max(best_at.get((0, k), -1), best_at.get((1, k), -1))

random.seed(1)
found = None
for _ in range(5000):
    nums = [random.randint(0, 6) for _ in range(random.randint(2, 7))]
    k, lim = random.randint(-5, 5), random.randint(1, 40)
    if wrong_keep_max(nums, k, lim) != max_product_brute(nums, k, lim):
        found = (nums, k, lim); break
print('phản ví dụ:', found, '| sai:', wrong_keep_max(*found), '| đúng:', max_product_brute(*found))

phản ví dụ: ([4, 1, 6, 0, 6, 4], -1, 3) | sai: -1 | đúng: 0


**Vì sao sai:** giữ tích lớn có thể khiến bước sau vượt `limit`, trong khi tích nhỏ hơn lại hợp lệ (mục 3.4).

### Lỗi 2 — Vứt bỏ hẳn các tích vượt `limit` (quên số 0)

In [16]:
def wrong_drop_over(nums, k, limit):
    if abs(k) > sum(nums):
        return -1
    dp = {}
    for x in nums:
        new = {}
        if x <= limit:
            new[(1, x)] = {x}
        for (p, s), prods in dp.items():
            key = (1 - p, s + x if p == 0 else s - x)
            new.setdefault(key, set()).update(q for q in (pr * x for pr in prods) if q <= limit)
        for key, v in new.items():
            dp.setdefault(key, set()).update(v)
    return max([pr for p in (0, 1) for pr in dp.get((p, k), ())], default=-1)

case = ([12, 0], 12, 5)             # [12, 0]: tổng 12 - 0 = 12, tích 0
print('sai :', wrong_drop_over(*case), '| đúng:', Solution().maxProduct(*case))

sai : -1 | đúng: 0


**Vì sao sai:** `[12]` có tích 12 > 5 bị bỏ; nhưng `[12, 0]` có tổng 12 và tích 0 ≤ 5. Trạng thái `OVER` cần được giữ để nhân với 0.

### Lỗi 3 — Dấu xen kẽ theo **chỉ số trong mảng gốc** thay vì trong dãy con

In [17]:
def wrong_sign_by_index(nums, k, limit):
    n, best = len(nums), -1
    for mask in range(1, 1 << n):
        idx = [i for i in range(n) if mask >> i & 1]
        s = sum(nums[i] if i % 2 == 0 else -nums[i] for i in idx)      # SAI
        if s == k:
            p = 1
            for i in idx:
                p *= nums[i]
            if p <= limit:
                best = max(best, p)
    return best

print('sai :', wrong_sign_by_index([1, 2, 3], 2, 10), '| đúng:', Solution().maxProduct([1, 2, 3], 2, 10))
print('sai :', wrong_sign_by_index([4, 1, 2], 2, 10), '| đúng:', Solution().maxProduct([4, 1, 2], 2, 10))

sai : 6 | đúng: 6
sai : 2 | đúng: 8


**Vì sao sai:** dấu do **vị trí trong dãy con** quyết định: `[1, 3]` (chọn chỉ số 0 và 2) có tổng `1 − 3`, không phải `1 + 3`.

### Lỗi 4 — Quên ràng buộc "khác rỗng" (coi dãy rỗng tổng 0, tích 1)

In [18]:
def wrong_allow_empty(nums, k, limit):
    r = Solution().maxProduct(nums, k, limit)
    return max(r, 1) if k == 0 and limit >= 1 else r          # SAI: dãy rỗng có "tích 1"

print('sai :', wrong_allow_empty([5, 3], 0, 10), '| đúng:', Solution().maxProduct([5, 3], 0, 10))

sai : 1 | đúng: -1


**Vì sao sai:** dãy con phải khác rỗng; `[5, 3]` không có dãy con nào tổng 0 ⇒ `−1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối ưu một đại lượng nhưng có **trần** (≤ limit)" ⇒ thường phải lưu **tập** giá trị đạt được, không chỉ max.
- "Tổng xen kẽ của dãy con" ⇒ trạng thái chẵn lẻ độ dài.
- Giá trị nhỏ (≤ 12) và trần nhỏ (≤ 5000) ⇒ số tích phân biệt nhỏ ⇒ DP tập khả thi.
- Số `0` và số `1` là trường hợp đặc biệt của phép nhân — xử lý riêng cho nhanh và đúng.

## 15. Biến thể của bài

1. Tích **nhỏ nhất ≥ limit** ⇒ giữ tập tích (chặn trên bởi một giá trị lớn), lấy min thỏa.
2. Đếm **số dãy con** có tổng xen kẽ `k` ⇒ DP đếm `(p, s)` (không cần tích).
3. Thay tích bằng **XOR** các phần tử ⇒ tập XOR (≤ 16 giá trị với số ≤ 12).
4. Bài liên quan: LC 1911 *Maximum Alternating Subsequence Sum*, LC 494 *Target Sum*.

In [19]:
# Biến thể 2: đếm số dãy con có tổng xen kẽ k
def count_alt(nums, k):
    cnt = {}
    for x in nums:
        new = {(1, x): 1}
        for (p, s), c in cnt.items():
            key = (1 - p, s + x if p == 0 else s - x)
            new[key] = new.get(key, 0) + c
        for key, c in new.items():
            cnt[key] = cnt.get(key, 0) + c
    return cnt.get((0, k), 0) + cnt.get((1, k), 0)

nums = [2, 2, 3, 3]
print(count_alt(nums, 0), sum(1 for m in range(1, 16) if alt_sum([nums[i] for i in range(4) if m >> i & 1]) == 0))

3 3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1 | Vét cạn | `O(2^n n)` | Oracle |
| 2 | Tập bộ ba | đa thức | Trực quan |
| 3 | `dict → set` | đa thức, nhanh | **Nộp bài** |
| 4 | Tách số 0 | đa thức | Lập luận rõ ràng về số 0 |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `maxProduct([2, 1, 2], 3, 10)`.

**Bài 2 (tính tay):** `maxProduct([3, 1, 2, 4], 0, 20)`.

**Bài 3 (điền TODO):** viết `cap(v, limit)` trả về `v` hoặc `OVER = limit + 1`.

**Bài 4 (tự viết lại):** biến thể 1 — tích **nhỏ nhất** `≥ limit` (trả `−1` nếu không có).

**Bài 5 (mini challenge):** trả về **một dãy con** đạt đáp án (danh sách chỉ số).

In [20]:
def cap_todo(v, limit):
    # TODO
    raise NotImplementedError

def min_product_at_least_todo(nums, k, limit):
    # TODO
    raise NotImplementedError

def witness_todo(nums, k, limit):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Dãy con tổng 3: `[2,1,2]` (`2−1+2 = 3`, tích 4). (2) Không còn dãy nào khác tổng 3? `[1,2]` cho `−1`. (3) `4`.

**Bài 2.** (1) Liệt kê dãy con có tổng xen kẽ 0: `[3,1,2,4]` (`3−1+2−4 = 0`, tích 24 > 20). (2) Không còn dãy con nào khác có tổng 0 (hai phần tử phải bằng nhau — không có). (3) Đáp án `−1`.

**Bài 3.** (1) `v ≤ limit` ⇒ `v`. (2) Ngược lại `limit + 1`. (3) Dùng trong DP.

**Bài 4.** (1) Không chặn tích (đáp án có thể lớn). (2) Lấy min các tích `≥ limit` ở trạng thái tổng `k`. (3) Vét cạn kiểm tra.

**Bài 5.** (1) Lưu "cha" cho mỗi `(p, s, prod)`. (2) Lần ngược. (3) Kiểm tra tổng và tích.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
a1 = Solution().maxProduct([2, 1, 2], 3, 10)
a2 = Solution().maxProduct([3, 1, 2, 4], 0, 20)
assert a1 == 4 == max_product_brute([2, 1, 2], 3, 10)
assert a2 == -1 == max_product_brute([3, 1, 2, 4], 0, 20)

def cap(v, limit):
    return v if v <= limit else limit + 1

assert cap(5, 5) == 5 and cap(6, 5) == 6 and cap(7, 5) == 6
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3: đúng')

Bài 1: 4 | Bài 2: -1 | Bài 3: đúng


In [22]:
# Bài 4: tích nhỏ nhất >= limit (giữ tích CHÍNH XÁC — chỉ phù hợp n nhỏ như bài tập)
def min_product_at_least(nums, k, limit):
    if abs(k) > sum(nums):
        return -1
    dp = {}
    for x in nums:
        new = {(1, x): {x}}
        for (p, s), prods in dp.items():
            key = (1 - p, s + x if p == 0 else s - x)
            new.setdefault(key, set()).update(pr * x for pr in prods)
        for key, v in new.items():
            dp.setdefault(key, set()).update(v)
    cands = [pr for p in (0, 1) for pr in dp.get((p, k), ()) if pr >= limit]
    return min(cands) if cands else -1

def min_brute(nums, k, limit):
    n, best = len(nums), None
    for mask in range(1, 1 << n):
        seq = [nums[i] for i in range(n) if mask >> i & 1]
        if alt_sum(seq) == k:
            p = 1
            for v in seq:
                p *= v
            if p >= limit and (best is None or p < best):
                best = p
    return -1 if best is None else best

random.seed(4)
for _ in range(300):
    nums = [random.randint(0, 12) for _ in range(random.randint(1, 9))]
    k, lim = random.randint(-10, 10), random.randint(1, 200)
    assert min_product_at_least(nums, k, lim) == min_brute(nums, k, lim), (nums, k, lim)
assert min_product_at_least([12, 12], 0, 1) == 144
print('Bài 4: khớp vét cạn; [12, 12], k = 0, limit = 1 ->', min_product_at_least([12, 12], 0, 1))

Bài 4: khớp vét cạn; [12, 12], k = 0, limit = 1 -> 144


Ở Bài 4 ta giữ tích **chính xác** (không chặn) vì đáp án có thể lớn tùy ý (ví dụ `[12, 12]`, `k = 0`, `limit = 1` ⇒ 144). Cách này chỉ phù hợp `n` nhỏ; với `n = 150` số tích phân biệt có thể bùng nổ.

In [23]:
# Bài 5: trả về một dãy con đạt đáp án
def witness(nums, k, limit):
    OVER = limit + 1
    states = {}                                         # (p, s, prod) -> danh sách chỉ số
    for i, x in enumerate(nums):
        new = {}
        st = (1, x, x if x <= limit else OVER)
        new.setdefault(st, [i])
        for (p, s, pr), idx in states.items():
            q = 0 if x == 0 else (OVER if pr == OVER or pr * x > limit else pr * x)
            st = (1 - p, s + x if p == 0 else s - x, q)
            new.setdefault(st, idx + [i])
        for st, idx in new.items():
            states.setdefault(st, idx)
    best = None
    for (p, s, pr), idx in states.items():
        if s == k and pr <= limit and (best is None or pr > best[0]):
            best = (pr, idx)
    return best

for nums, k, lim, exp in examples:
    w = witness(nums, k, lim)
    if w is None:
        assert exp == -1
    else:
        pr, idx = w
        seq = [nums[i] for i in idx]
        p = 1
        for v in seq:
            p *= v
        assert pr == exp == p and alt_sum(seq) == k
    print(nums, '->', w)

[1, 2, 3] -> (6, [0, 1, 2])
[0, 2, 3] -> None
[2, 2, 3, 3] -> (9, [2, 3])


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
nếu |k| > sum(nums): return -1
OVER = limit + 1
dp[(p, s)] = set(tích)          # p: chẵn lẻ độ dài
for x in nums:
    new[(1, x)] = {cap(x)}
    for (p, s), P in dp: key = (1-p, s + x if p == 0 else s - x)
        x == 0 -> {0};  x == 1 -> P;  else {cap(pr * x)}
    gộp new vào dp
đáp án = max tích <= limit trong dp[(0,k)] ∪ dp[(1,k)], hoặc -1
```

**Checklist:**

- [ ] Lưu **tập** tích, không chỉ max?
- [ ] Giữ trạng thái `OVER` để nhân với 0?
- [ ] Dấu theo vị trí **trong dãy con**?
- [ ] Dãy con **khác rỗng**?
- [ ] Cắt sớm khi `|k| > Σ nums`?